![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# XAI for Transformers: <Method's title>


This Notebook shows ... 

--------

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. To start working on the notebook, click on the following button. This will open this page in the Colab environment, and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/xai-for-transformer/2-Tutorial_AttentionMaps_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Google Colab follow the next step:

1. Run this cell to connect your Google Drive to Colab and install packages
2. Allow this notebook to access your Google Drive files. Click on 'Yes', and select your account.
3. "Google Drive for desktop wants to access your Google Account". Click on 'Allow'.
   
A folder has been created in your Drive, and you can navigate it through the lefthand panel in Colab. You might also receive an email that informs you about the access on your Google Drive.

In [1]:
# Mount drive folder to dbe abale to download repo
# from google.colab import drive
# drive.mount('/content/drive')

# Switch to correct folder'
# %cd /content/drive/MyDrive

In [2]:
# Don't run this cell if you already cloned the repo 
# %rm -r XAI-Tutorials
# !git clone --branch main https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials.git

In [3]:
# Install al required dependencies and package versions
# %cd XAI-Tutorials
# !pip install -r requirements_xai-for-transformer.txt
# %cd xai-for-transformer

### Imports

In [1]:
import torch
import os
import zipfile
import requests

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# add your other imports here

# Select the best available device: NVIDIA GPU (CUDA), Apple Silicon GPU (MPS), or CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


---

## Model Loading

In the notebook [*1-Tutorial_Transformer_Model.ipynb*](./1-Tutorial_Transformer_Model.ipynb), we introduced the Transformer architecture and the DistilBERT model used throughout this tutorial series.

Rather than training a model from scratch, we use the pretrained **DistilBERT** model (`distilbert-base-uncased`) with a sequence classification head configured for six emotion classes. At this stage, the classification head is randomly initialized, allowing us to develop and test the XAI pipeline (e.g., SHAP, LIME, and Integrated Gradients) independently of model performance.

Note: Once our own DistilBERT model has been fine-tuned on the emotion dataset, we can simply replace the pretrained checkpoint with the fine-tuned model while keeping the rest of the explainability workflow unchanged.


In [2]:
# Fine-tuned DistilBERT emotion weights, hosted as a GitHub Release asset.
# NOTE: the download link becomes active once the weights Release is published.
url = "https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/releases/download/distilbert-emotion-weights/distilbert_emotion_weights.zip"

weights_path = "../data/distilbert-emotion"
zip_path = "../data/distilbert_emotion_weights.zip"
os.makedirs("../data", exist_ok=True)

if not os.path.exists(weights_path):
    # Reuse a local zip if it is already there (e.g. shared for testing),
    # otherwise download it from the Release.
    if not os.path.exists(zip_path):
        with open(zip_path, "wb") as f:
            f.write(requests.get(url).content)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("../data/")

model = AutoModelForSequenceClassification.from_pretrained(weights_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(weights_path)

## XAI Method Name
Please remember to add at least:
- method explanation (consider recording a short video 3-5 minutes)
- reference
- method implementation
- examples and comments
- 1-3 open questions and the respective answers
